# Notebook baseline: abandono de clientes (churn)

Este es el **punto de partida** del proyecto: un notebook típico de data science que
funciona, pero que todavía no es reproducible ni reutilizable fuera de aquí.

En la **práctica 1** lo recorrerás de arriba abajo e identificarás:

1. **Entradas** (columnas) y **target**.
2. **Preprocesamiento** (qué transformaciones se aprenden de los datos).
3. **Entrenamiento** y **evaluación**.
4. **Salidas** (métricas, modelo, predicciones).

Después moverás esas piezas a `src/mlops_course/`.

> Ejecuta siempre con *Kernel → Restart Kernel and Run All Cells*: si falla desde un
> kernel limpio, no es reproducible.

## 1. Carga de datos

In [48]:
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

SEMILLA = 42
df = pd.read_csv("../data/churn.csv")
print(df.shape)
df.head()

(7043, 7)


,id_cliente,antiguedad_meses,cuota_mensual,tipo_contrato,tipo_internet,metodo_pago,abandona
0,7590-VHVEG,1,29.85,mensual,dsl,cheque_electronico,0
1,5575-GNVDE,34,56.95,anual,dsl,cheque_postal,0
2,3668-QPYBK,2,53.85,mensual,dsl,cheque_postal,1
3,7795-CFOCW,45,42.30,anual,dsl,transferencia,0
4,9237-HQITU,2,70.70,mensual,fibra,cheque_electronico,1


Diccionario de datos (ver `data/README.md`): cada fila es un cliente.
`abandona = 1` indica que el cliente se dio de baja.

In [49]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 7 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   id_cliente        7043 non-null   str    
 1   antiguedad_meses  7043 non-null   int64  
 2   cuota_mensual     7043 non-null   float64
 3   tipo_contrato     7043 non-null   str    
 4   tipo_internet     7043 non-null   str    
 5   metodo_pago       7043 non-null   str    
 6   abandona          7043 non-null   int64  
dtypes: float64(1), int64(2), str(4)
memory usage: 631.5 KB


In [50]:
df["abandona"].value_counts(normalize=True).round(3)

abandona
0    0.735
1    0.265
Name: proportion, dtype: float64

In [51]:
df["abandona"].value_counts(normalize=False).round(3)

abandona
0    5174
1    1869
Name: count, dtype: int64

In [52]:
df.describe().transpose()

,count,mean,std,min,25%,50%,75%,max
antiguedad_meses,7043.0,32.371149,24.559481,0.00,9.0,29.00,55.00,72.00
cuota_mensual,7043.0,64.761692,30.090047,18.25,35.5,70.35,89.85,118.75
abandona,7043.0,0.265370,0.441561,0.00,0.0,0.00,1.00,1.00


In [53]:
df.describe(include="all").T

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
id_cliente,7043,7043,7590-VHVEG,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
antiguedad_meses,7043.0,NaN,NaN,NaN,32.371149,24.559481,0.0,9.0,29.0,55.0,72.0
cuota_mensual,7043.0,NaN,NaN,NaN,64.761692,30.090047,18.25,35.5,70.35,89.85,118.75
tipo_contrato,7043,3,mensual,3875,NaN,NaN,NaN,NaN,NaN,NaN,NaN
tipo_internet,7043,3,fibra,3096,NaN,NaN,NaN,NaN,NaN,NaN,NaN
metodo_pago,7043,4,cheque_electronico,2365,NaN,NaN,NaN,NaN,NaN,NaN,NaN
abandona,7043.0,NaN,NaN,NaN,0.26537,0.441561,0.0,0.0,0.0,1.0,1.0


In [54]:
df.describe(include="str").T

,count,unique,top,freq
id_cliente,7043,7043,7590-VHVEG,1
tipo_contrato,7043,3,mensual,3875
tipo_internet,7043,3,fibra,3096
metodo_pago,7043,4,cheque_electronico,2365


In [55]:
df['tipo_internet'].value_counts(normalize=True).round(4)

tipo_internet
fibra           0.4396
dsl             0.3437
sin_internet    0.2167
Name: proportion, dtype: float64

In [56]:
# Filas duplicadas
df.duplicated().sum()
# df.isna().sum()

np.int64(0)

## 2. Entradas y target

`id_cliente` es un identificador: **no** es una variable predictiva.

In [57]:
NUMERICAS = ["antiguedad_meses", "cuota_mensual"]
CATEGORICAS = ["tipo_contrato", "tipo_internet", "metodo_pago"]
TARGET = "abandona"

X = df[NUMERICAS + CATEGORICAS]
y = df[TARGET]

## 3. Split reproducible

Primero se separa el **test** (no se toca hasta el final) y luego **validación**.
`stratify` mantiene la proporción de abandonos en cada partición.

In [58]:
X_resto, X_test, y_resto, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=SEMILLA
)
X_train, X_val, y_train, y_val = train_test_split(
    X_resto, y_resto, test_size=0.2, stratify=y_resto, random_state=SEMILLA
)

print(f"train: {len(X_train)}, validación: {len(X_val)}, test: {len(X_test)}")

train: 4507, validación: 1127, test: 1409


## 4. Preprocesamiento + modelo en un único `Pipeline`

El escalado y el one-hot **se ajustan solo con train** al llamar a `fit`: así se evita
fuga de información (*data leakage*) y el mismo objeto sirve para predecir después.

In [ ]:
preprocesamiento = ColumnTransformer(
    [
        ("numericas", StandardScaler(), NUMERICAS),
        ("categoricas", OneHotEncoder(handle_unknown="ignore"), CATEGORICAS),
    ]
)

modelo = Pipeline(
    [
        ("preprocesamiento", preprocesamiento),
        ("clasificador", LogisticRegression(C=1.0, max_iter=1000)),
    ]
)
modelo.fit(X_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocesamiento', ...), ('clasificador', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels. Only exist if the last step of the pipeline is aclassifier.","ndarray[int64](2,)","[0,1]"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](5,)","['antiguedad_meses','cuota_mensual','tipo_contrato','tipo_internet', 'metodo_pago']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,5
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('numericas', ...), ('categoricas', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifyin

## 5. Evaluación en validación

In [60]:
modelo.predict(X_val)

array([0, 0, 0, ..., 0, 0, 0], shape=(1127,))

In [61]:
# (modelo.predict_proba(X_val)[:,1] >= 0.3)

In [62]:
# modelo.predict_proba(X_val) #[:,1]

In [63]:
UMBRAL = 0.5
proba_val = modelo.predict_proba(X_val)[:, 1]
pred_val = (proba_val >= UMBRAL).astype(int)
print(f"ROC AUC validación: {roc_auc_score(y_val, proba_val):.3f}")
print(classification_report(y_val, pred_val, digits=3))

ROC AUC validación: 0.840
              precision    recall  f1-score   support

           0      0.839     0.920     0.878       828
           1      0.699     0.512     0.591       299

    accuracy                          0.812      1127
   macro avg      0.769     0.716     0.734      1127
weighted avg      0.802     0.812     0.802      1127



**Salida esperada (orientativa):** ROC AUC de validación en torno a 0,83–0,85.
Pequeñas diferencias entre equipos son normales.

## 6. Predicción para clientes nuevos

In [64]:
nuevos = pd.read_csv("../data/churn_nuevos.csv")
nuevos.head()

,id_cliente,antiguedad_meses,cuota_mensual,tipo_contrato,tipo_internet,metodo_pago
0,9259-PACGQ,34,62.65,mensual,dsl,cheque_electronico
1,5914-DVBWJ,18,85.45,mensual,fibra,cheque_electronico
2,9734-YWGEX,9,98.30,mensual,fibra,tarjeta
3,5035-PGZXH,56,106.80,anual,fibra,cheque_electronico
4,6362-QHAFM,42,108.30,anual,fibra,cheque_electronico


In [66]:
nuevos["probabilidad"] = modelo.predict_proba(nuevos[NUMERICAS + CATEGORICAS])[:, 1]
nuevos["abandona"] = (nuevos["probabilidad"] >= UMBRAL).astype(int)
nuevos.tail()

,id_cliente,antiguedad_meses,cuota_mensual,tipo_contrato,tipo_internet,metodo_pago,probabilidad,abandona
45,6754-WKSHP,30,25.35,anual,sin_internet,transferencia,0.041148,0
46,4102-HLENU,67,65.65,bianual,dsl,cheque_postal,0.015463,0
47,6996-KNSML,1,74.90,mensual,fibra,cheque_electronico,0.722123,1
48,7666-WKRON,39,80.45,bianual,fibra,cheque_electronico,0.132336,0
49,2676-ISHSF,1,20.30,mensual,sin_internet,cheque_postal,0.191918,0


## Preguntas para la práctica 1

- ¿Qué valores están *hardcodeados* (rutas, semilla, umbral, hiperparámetros)?
  ¿Dónde deberían vivir?
- ¿Qué pasaría si mañana alguien quiere predecir sin abrir este notebook?
- ¿Qué parte se aprende de los datos y qué parte es configuración?
- ¿Dónde se guarda el modelo entrenado? (pista: en ningún sitio, todavía)